# ARC-AGI-3 public-agent corpus builder
This notebook resolves the attached v3 builder before any project import, parses attached public notebooks without executing them, and writes a validated corpus to `/kaggle/working/arc_agi3_agent_corpus`.

In [ ]:
from __future__ import annotations

import os
import json
import sys
import tarfile
import zipfile
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')
RUNTIME_ROOT = WORK_ROOT / 'arc_corpus_runtime'
RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)

def safe_target(root: Path, member_name: str) -> Path:
    target = (root / member_name).resolve()
    if root.resolve() not in target.parents and target != root.resolve():
        raise ValueError(f'Archive path traversal rejected: {member_name}')
    return target

def extract_archive(archive: Path, destination: Path) -> None:
    if archive.suffix.lower() == '.zip':
        with zipfile.ZipFile(archive) as handle:
            for member in handle.infolist():
                safe_target(destination, member.filename)
            handle.extractall(destination)
        return
    if archive.name.endswith(('.tar.gz', '.tgz', '.tar')):
        with tarfile.open(archive) as handle:
            for member in handle.getmembers():
                safe_target(destination, member.name)
                if member.issym() or member.islnk():
                    raise ValueError(f'Archive link rejected: {member.name}')
            handle.extractall(destination, filter='data')
        return
    raise ValueError(f'Unsupported archive: {archive}')

def find_project() -> Path:
    roots = [Path.cwd(), *sorted(INPUT_ROOT.glob('**/pyproject.toml'))]
    for candidate in roots:
        root = candidate if candidate.is_dir() else candidate.parent
        if (root / 'src' / 'arc_corpus' / '__init__.py').exists():
            return root
    archives = sorted(INPUT_ROOT.glob('**/*.zip')) + sorted(INPUT_ROOT.glob('**/*.tar.gz')) + sorted(INPUT_ROOT.glob('**/*.tgz'))
    for archive in archives:
        destination = RUNTIME_ROOT / archive.stem.replace('.tar', '')
        destination.mkdir(parents=True, exist_ok=True)
        extract_archive(archive, destination)
        matches = sorted(destination.glob('**/src/arc_corpus/__init__.py'))
        if matches:
            return matches[0].parents[2]
    raise FileNotFoundError('Attach the arc-agi3-agent-corpus source tree or archive as a Kaggle input.')

PROJECT_ROOT = find_project()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
resolver_report = {
    'project_root': str(PROJECT_ROOT),
    'resolver': 'v3-early-input-resolver',
    'input_root': str(INPUT_ROOT),
}
report_path = WORK_ROOT / 'data' / 'reports' / 'arc_import_resolver.json'
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(json.dumps(resolver_report, indent=2) + '\n', encoding='utf-8')
print({**resolver_report, 'report': str(report_path)})

In [ ]:
from arc_corpus.pipeline import BuildOptions, build_corpus

CATALOG = Path(os.environ.get('ARC_CORPUS_CATALOG', PROJECT_ROOT / 'config' / 'seed_public_notebooks.json'))
RAW_DIR = Path(os.environ.get('ARC_CORPUS_RAW_DIR', '/kaggle/input/arc-agi3-public-notebooks'))
OUTPUT_DIR = WORK_ROOT / 'arc_agi3_agent_corpus'

options = BuildOptions(
    catalog_path=CATALOG,
    raw_dir=RAW_DIR,
    output_dir=OUTPUT_DIR,
    fetch_missing=os.environ.get('ARC_CORPUS_FETCH_MISSING', '0') == '1',
    min_quality=float(os.environ.get('ARC_CORPUS_MIN_QUALITY', '0.35')),
    include_cells=os.environ.get('ARC_CORPUS_COMPONENTS_ONLY', '0') != '1',
)
manifest = build_corpus(options)
print({
    'corpus_sha256': manifest['corpus_sha256'],
    'sources': manifest['source_counts'],
    'records': manifest['record_counts'],
    'validation': manifest['validation'],
})

In [ ]:
import shutil

archive_path = shutil.make_archive(str(WORK_ROOT / 'arc_agi3_agent_corpus'), 'zip', OUTPUT_DIR)
print({'archive': archive_path, 'bytes': Path(archive_path).stat().st_size})